## 📥 Load Tweets from Twitter API (or CSV if already collected)

In [ ]:

import pandas as pd

# Load tweet data collected via API (or replace this with actual API call output)
# If you already have CSV:
df = pd.read_csv("ai_leadership_tweets.csv")

df.head()


## 🧽 Clean Tweet Text

In [ ]:

import re

def clean_tweet(text):
    text = re.sub(r"http\S+", "", text)           # Remove URLs
    text = re.sub(r"@\w+", "", text)              # Remove mentions
    text = re.sub(r"#", "", text)                  # Remove hashtags symbols
    text = re.sub(r"RT[\s]+", "", text)           # Remove retweet indicators
    text = re.sub(r"[^\w\s.,!?]", "", text)        # Remove emojis/special characters
    text = re.sub(r"\s+", " ", text)              # Normalize whitespace
    return text.lower().strip()

df['clean_text'] = df['text'].apply(clean_tweet)
df[['text', 'clean_text']].head()


## 💬 TextBlob Sentiment Analysis

In [ ]:

from textblob import TextBlob

def textblob_sentiment(text):
    return TextBlob(text).sentiment.polarity

df['textblob_polarity'] = df['clean_text'].apply(textblob_sentiment)

df['textblob_label'] = df['textblob_polarity'].apply(
    lambda x: 'positive' if x > 0.05 else ('negative' if x < -0.05 else 'neutral')
)

df[['clean_text', 'textblob_polarity', 'textblob_label']].head()


## 🤖 BERT-Based Sentiment Classification

In [ ]:

!pip install transformers --quiet

from transformers import pipeline
sentiment_model = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

df['bert_sentiment'] = df['clean_text'].apply(lambda x: sentiment_model(x)[0]['label'])
df[['clean_text', 'bert_sentiment']].head()


## ❤️ Emotion Detection with RoBERTa

In [ ]:

!pip install transformers --quiet

emotion_model = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=False)
df['emotion'] = df['clean_text'].apply(lambda x: emotion_model(x)[0]['label'])
df[['clean_text', 'emotion']].head()


## 🧠 Topic Modeling with BERTopic

In [ ]:

!pip install bertopic umap-learn --quiet

from bertopic import BERTopic

topic_model = BERTopic()
topics, _ = topic_model.fit_transform(df['clean_text'].astype(str).tolist())
df['topic'] = topics
topic_model.get_topic_info().head()


## 🌐 Hashtag Co-Occurrence Network

In [ ]:

import networkx as nx

def extract_hashtags(text):
    return re.findall(r"#(\w+)", str(text).lower())

df['hashtags'] = df['text'].apply(extract_hashtags)

edges = []
for tags in df['hashtags']:
    for i in range(len(tags)):
        for j in range(i + 1, len(tags)):
            edges.append((tags[i], tags[j]))

G = nx.Graph()
G.add_edges_from(edges)

# Show degree centrality
centrality = nx.degree_centrality(G)
central_df = pd.DataFrame(centrality.items(), columns=['hashtag', 'degree_centrality'])
central_df.sort_values('degree_centrality', ascending=False).head()


## 💾 Save the Results

In [ ]:

df.to_csv("ai_leadership_full_analysis.csv", index=False)
print("Data saved to ai_leadership_full_analysis.csv")
